# P2 · Plan B2 probe: a diffusion prior maps label names into image space (checklist 1.6b)

A *prior* is the half of a DALL·E-2-style model that turns a CLIP **text** embedding into a CLIP **image**
embedding, with no picture in between. Kandinsky 2.1's prior predicts OpenAI CLIP **ViT-L/14** image
embeddings (no public prior exists for ViT-B/16), so this notebook works with the ViT-L/14 cache from your
Kaggle run `notebook1-vitl14` and exports, to `MyDrive/ReNeg/transfers/analysis_pack3/`:

1. `reneg_prior_probe.npz`: 4 prior embeddings for each P1 probe name (NINCO, SSB-hard, ImageNet).
2. `reneg_prior_pool.npz`: 2 prior embeddings for each of the 14,526 knowledge-graph names.
3. `reneg_l14_text.npz`: ViT-L/14 text embeddings of the ID, probe and pool names (the baseline).
4. `reneg_l14_centroids.npz`: ViT-L/14 image centroids of every NINCO, SSB-hard and ImageNet class.
5. `reneg_l14_to_b16_map.npz`: an image-to-image map from ViT-L/14 to ViT-B/16, fitted on ImageNet val
   images encoded by both, and checked on NINCO and SSB-hard images it never saw.

**Runtime:** GPU (A100 / L4 / T4). About 15 minutes for the probe, plus 20–40 minutes for the pool names
(resumable). The first run also downloads your Kaggle ViT-L/14 cache (about 0.6 GB).

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True)
ctx = start("P2_prior_probe", need_gpu=False)
log = ctx.log
import json, subprocess, time
import numpy as np, torch
from reneg import colab as rc
if not torch.cuda.is_available():
    print("No GPU: Runtime -> Change runtime type -> A100, L4 or T4 (the prior is slow on a CPU)")
L = rc.layout(DRIVE_ROOT)
OUT = f"{DRIVE_ROOT}/transfers/analysis_pack3"
os.makedirs(OUT, exist_ok=True)
KAGGLE_SLUG = "notebook1-vitl14"          # your Kaggle copy of notebook1 that built the ViT-L/14 cache

## 1 · The ViT-L/14 cache (downloaded from Kaggle on the first run)

In [ ]:
have = os.path.isfile(os.path.join(L["cache"], "vit-l-14", "textbank.pt"))
if not have:
    user = rc.setup_kaggle_credentials()
    if user:
        rc.ensure_kaggle_cli()
        slug = rc.resolve_or_guess({"01L": KAGGLE_SLUG}, rc.list_my_kernels())["01L"]
        d = os.path.join(L["transfers"], "kaggle", slug)
        ok = rc.download_kernel_output(f"{user}/{slug}", d)
        print(f"{slug}: {'downloaded' if ok else 'FAILED -> download its output zip on Kaggle and put it in uploads/'}")
    rc.unpack_uploads(L)
    tags = rc.install_cache([os.path.join(L["transfers"], "kaggle"), L["uploads"]], L["cache"])
    print("caches installed:", tags or "nothing new")
from oodlab.report import find_cache
cache_l = find_cache(ctx, "ViT-L/14")
cache_b = find_cache(ctx, "ViT-B/16")
bank_l = cache_l.load_textbank()
print("ViT-L/14 sets:", cache_l.available())

## 2 · Load the prior and check that it speaks our ViT-L/14

In [ ]:
from oodlab.clipwrap import load_clip
from reneg.g1 import clip_text_encoder
from reneg.prior import load_kandinsky_prior, image_encoder_agreement, DEFAULT_PRIOR
FAKE = os.environ.get("RENEG_FAKE_GEN") == "1"          # offline tests of this notebook only
WEIGHTS = f"{DRIVE_ROOT}/weights"
model = load_clip("ViT-L/14", device=ctx.device, input_roots=ctx.input_roots + [WEIGHTS], download_root=WEIGHTS,
                  random_init=FAKE)
enc_text = clip_text_encoder(model, prompt="The nice {}.")
if FAKE:
    from reneg.prior import fake_prior
    prior = fake_prior(int(bank_l.id_text.shape[1]))
    agree = 1.0
else:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            os.environ["HF_TOKEN"] = tok
    except Exception:
        pass
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.27", "transformers", "accelerate"],
                   check=False)
    prior = load_kandinsky_prior(DEFAULT_PRIOR, device=ctx.device, log=log.info)
    agree = image_encoder_agreement(prior.pipe, model, device=ctx.device)
print(f"prior's image encoder vs our ViT-L/14: min cosine {agree:.4f} (should be above 0.99)")

## 3 · Real ViT-L/14 centroids and the guidance setting (chosen on ImageNet classes only)

In [ ]:
from reneg.concepts import folder_of
from reneg.prior import concept_centroids, run_prior, load_prior_embeddings
from reneg.transport import l2n
cents = {}
for ds in ("ninco", "ssb_hard"):
    fs = cache_l.load_set(ds)
    f = np.asarray([folder_of(k) for k in fs.keys])
    uniq = sorted(set(f))
    idx = {u: i for i, u in enumerate(uniq)}
    g = np.asarray([idx[u] for u in f])
    A, B = concept_centroids(fs.feats, g, len(uniq), split=(np.arange(len(g)) % 2 == 0))
    cents[ds] = (uniq, A, B)
fs = cache_l.load_set("imagenet_test")
y = np.asarray(fs.labels).astype(int)
A, B = concept_centroids(fs.feats, y, int(bank_l.id_text.shape[0]), split=(np.arange(len(y)) % 2 == 0))
cents["imagenet"] = ([str(i) for i in range(A.shape[0])], A, B)
np.savez_compressed(os.path.join(OUT, "reneg_l14_centroids.npz"),
                    **{f"{k}__folders": np.asarray(v[0], dtype=str) for k, v in cents.items()},
                    **{f"{k}__A": v[1].half().numpy() for k, v in cents.items()},
                    **{f"{k}__B": v[2].half().numpy() for k, v in cents.items()})
# probe names = the P1 names (same 414 concepts), so P1 and P2 compare directly
p1 = f"{DRIVE_ROOT}/transfers/analysis_pack2/reneg_gen_probe.npz"
if os.path.isfile(p1):
    z = np.load(p1)
    names, tags, folders = list(z["names"]), list(z["tags"]), list(z["folders"])
else:
    from reneg.gen import probe_names
    from reneg.wordnet import get_wordnet
    names, tags, folders = probe_names(cache_b, cache_b.load_textbank(), wn=get_wordnet())
print({t: tags.count(t) for t in dict.fromkeys(tags)}, "probe names")
im = [i for i, t in enumerate(tags) if t == "imagenet"]
real = cents["imagenet"][2]
truth = np.asarray([int(folders[i]) for i in im])
best = None
for gscale in (1.0, 4.0):
    E = prior([f"a photo of a {names[i]}." for i in im], n_per=4, seed=1, guidance=gscale)
    P = l2n(E.reshape(len(im), 4, -1).mean(1))
    acc = 100 * float(((P @ real.T).argmax(1).numpy() == truth).mean())
    print(f"guidance {gscale}: prior prototype finds its ImageNet class (of 1000) {acc:.1f}% of the time")
    if best is None or acc > best[1]:
        best = (gscale, acc)
T = l2n(enc_text([names[i] for i in im]).float())
print(f"text embedding finds its class {100 * float(((T @ real.T).argmax(1).numpy() == truth).mean()):.1f}% of the time")
GUIDANCE = best[0]
print("guidance used from here on:", GUIDANCE)

## 4 · Prior embeddings for the probe names and for the knowledge-graph pool (resumable)

In [ ]:
p_probe = os.path.join(OUT, "reneg_prior_probe.npz")
run_prior(prior, names, p_probe, n_per=4, batch=32, guidance=GUIDANCE, log=log.info)
from reneg.packs import load_kg_pool
pool_names = [r["name"] for r in load_kg_pool()["pool"]]
POOL = True                       # set False to skip the 14,526 pool names (20-40 minutes)
p_pool = os.path.join(OUT, "reneg_prior_pool.npz")
if POOL:
    run_prior(prior, pool_names, p_pool, n_per=2, batch=64, guidance=GUIDANCE, log=log.info)

## 5 · ViT-L/14 text embeddings (the baseline) and the ViT-L/14 → ViT-B/16 image map

In [ ]:
from reneg.packs import q8
from reneg.prior import align_by_keys, apply_image_map, fit_image_map
texts = {"id": l2n(bank_l.id_text.float()), "probe": l2n(enc_text(names).float())}
texts["pool"] = l2n(torch.cat([enc_text(pool_names[s:s + 2000]).float() for s in range(0, len(pool_names), 2000)]))
arr = {}
for k, v in texts.items():
    q, s = q8(v)
    arr[f"{k}__q"], arr[f"{k}__s"] = q, s
np.savez_compressed(os.path.join(OUT, "reneg_l14_text.npz"), probe_names=np.asarray(names, dtype=str), **arr)
# image-to-image map, fitted on ImageNet val (both backbones encoded the same images)
a, b = cache_l.load_set("imagenet_val_all"), cache_b.load_set("imagenet_val_all")
ia, ib = align_by_keys(list(a.keys), list(b.keys))
perm = np.random.default_rng(0).permutation(len(ia))
tr, te = perm[: int(0.8 * len(perm))], perm[int(0.8 * len(perm)):]
XA, XB = a.feats[torch.as_tensor(ia)].float(), b.feats[torch.as_tensor(ib)].float()
checks = {}
best_map = None
for lam in (0.1, 1.0, 10.0):
    W = fit_image_map(XA[torch.as_tensor(tr)], XB[torch.as_tensor(tr)], lam)
    c = float((apply_image_map(W, XA[torch.as_tensor(te)]) * l2n(XB[torch.as_tensor(te)])).sum(1).mean())
    print(f"lambda {lam}: held-out ImageNet images, mean cosine(mapped ViT-L/14, real ViT-B/16) = {c:.4f}")
    if best_map is None or c > best_map[1]:
        best_map = (lam, c, W)
lam, c_in, W = best_map
checks["imagenet_heldout"] = c_in
for ds in ("ninco", "ssb_hard"):
    sa, sb = cache_l.load_set(ds), cache_b.load_set(ds)
    ja, jb = align_by_keys(list(sa.keys), list(sb.keys))
    cc = (apply_image_map(W, sa.feats[torch.as_tensor(ja)].float()) * l2n(sb.feats[torch.as_tensor(jb)].float())).sum(1)
    checks[ds] = float(cc.mean())
    print(f"{ds}: mean cosine(mapped, real ViT-B/16) on {len(ja)} images it never saw = {checks[ds]:.4f}")
np.savez_compressed(os.path.join(OUT, "reneg_l14_to_b16_map.npz"), W=W.float().numpy(), lam=np.asarray(lam),
                    checks=np.asarray(json.dumps(checks)))

## Files written

In [ ]:
files = sorted(os.path.join(OUT, f) for f in os.listdir(OUT) if f.endswith(".npz") and not f.endswith(".tmp.npz"))
for p in files:
    print(f"{os.path.getsize(p) / 1e6:6.1f} MB  {p}")
finish(ctx, {"files": [os.path.basename(p) for p in files], "guidance": GUIDANCE, "encoder_agreement": agree,
             "map_checks": checks})

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")